In [0]:
# --------------------------------------------------------------
# Set environment variables for the pipeline run
# --------------------------------------------------------------
# These values are retrieved from the master pipeline task 
# (superstore_pipeline_master_run_id_init) using dbutils.jobs.taskValues.
# They ensure that the ETL, Mart, Features, KPI, and Dashboard code
# runs in the correct environment (dev/qa/prod) without hardcoding paths.
# SUPERSTORE_ENV, SUPERSTORE_PIPELINE_NAME, and SUPERSTORE_PIPELINE_VERSION
# are set for downstream modules to dynamically resolve catalog/schema.
# --------------------------------------------------------------
import os

env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -------------------------------
# Environment Config
# -------------------------------
import sys
sys.path.append("/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/superstore_shared_utilities")
from superstore_platform_config import get_catalog, get_gold_schema, get_mart_schema

# Dynamically get environment-aware catalog and schemas
catalog = get_catalog()            # e.g., superstore_catalog_dev / _qa / _prod
gold_schema = get_gold_schema()    # e.g., superstore_gold
mart_schema = get_mart_schema() 

# -------------------------------
# Create mart Table (Environment-aware)
# -------------------------------
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{mart_schema}.mart_customer_360 AS
WITH customer_orders AS (
    SELECT
        c.customer_id,
        c.customer_name,
        c.segment,
        c.region,
        SUM(s.sales) AS total_sales,
        SUM(s.profit) AS total_profit,
        COUNT(DISTINCT o.order_id) AS total_orders,
        AVG(s.discount) AS discount_sensitivity,
        MAX(o.order_date) AS last_order_date,
        MIN(o.order_date) AS first_order_date
    FROM {catalog}.{gold_schema}.dim_customers c
    LEFT JOIN {catalog}.{gold_schema}.facts_sales s
        ON c.customer_id = s.customer_id
    LEFT JOIN {catalog}.{gold_schema}.facts_orders o
        ON c.customer_id = o.customer_id AND s.order_id = o.order_id
    WHERE c.is_current = TRUE
    GROUP BY c.customer_id, c.customer_name, c.segment, c.region
)
SELECT
    customer_id,
    customer_name,
    segment,
    region,
    total_sales,
    total_profit,
    total_orders,
    CASE WHEN total_orders = 0 THEN NULL ELSE ROUND(total_sales / total_orders, 2) END AS avg_order_value,
    last_order_date,
    DATEDIFF(current_date(), last_order_date) AS recency_days,
    DATEDIFF(last_order_date, first_order_date) AS customer_lifetime_days,
    discount_sensitivity,
    CASE WHEN total_profit > 0 THEN 1 ELSE 0 END AS profitability_flag
FROM customer_orders
""")

